

# 🚢 Metaflow — Tutorial Completo en Castellano
## Predicting Titanic Survival con MLOps de verdad

**¿Qué es Metaflow?**  
Metaflow es un framework de MLOps creado por Netflix (y ahora open-source) que te permite construir, versionar y ejecutar pipelines de Machine Learning de forma reproducible, escalable y ordenada.

**¿Qué aprenderás en este notebook?**

| Concepto | Descripción |
|---|---|
| `Flow` | La clase base que define un pipeline |
| `step` | Decorador que convierte un método en un paso del pipeline |
| `self` como estado compartido | Cómo pasar datos entre pasos |
| `FlowSpec` | La superclase de todo flow |
| `start` / `end` | Pasos obligatorios en todo flow |
| Branching | Ejecutar ramas paralelas |
| `@conda` / `@pypi` | Gestión de dependencias por paso |
| `@retry` | Reintentos automáticos ante fallos |
| `@timeout` | Límite de tiempo por paso |
| `@catch` | Manejo de errores |
| `@resources` | Asignación de CPU/memoria (para cloud) |
| `current` | Objeto con metadata del run en curso |
| `Client API` | Inspeccionar runs pasados desde el notebook |
| Artifacts | Datos persistidos automáticamente |
| Cards | Visualizaciones automáticas de runs |

---
> 💡 **Prerrequisito**: `pip install metaflow scikit-learn pandas seaborn`

## 0. Instalación y Setup

In [ ]:
# Instalamos todo lo necesario
!pip install metaflow scikit-learn pandas seaborn matplotlib --quiet

In [ ]:
# Comprobamos que metaflow está disponible
import metaflow
print(f"✅ Metaflow versión: {metaflow.__version__}")

# En Jupyter necesitamos esta línea mágica para que los flows funcionen
# dentro del notebook (evita conflictos con el event loop)
from metaflow import FlowSpec, step, Flow, Run, namespace, current
import pandas as pd
import numpy as np
print("✅ Imports OK")

---
## 1. Concepto Fundamental: ¿Qué es un Flow?

Un **Flow** es simplemente una clase Python que hereda de `FlowSpec`.  
Cada **método decorado con `@step`** es un paso del pipeline.  
Los pasos se conectan llamando a `self.next()`.

```
start  ──►  paso_A  ──►  paso_B  ──►  end
```

**Reglas de oro:**
1. Siempre debe existir un paso `start` y un paso `end`.
2. Cada paso debe terminar con `self.next(nombre_del_siguiente_paso)`.
3. El paso `end` no llama a `self.next()`.
4. Los datos se comparten entre pasos asignándolos a `self` (¡actúa como estado global del flow!).

---
## 2. El Flow más Simple del Mundo (Hello World)

In [ ]:
%%writefile flows/01_hola_metaflow.py
"""
CONCEPTO 1: FlowSpec, @step, start, end, self.next()

Este es el flow más básico posible. 
Sirve para entender la estructura mínima obligatoria.
"""

from metaflow import FlowSpec, step

class HolaMetaflow(FlowSpec):
    """
    El docstring de la clase describe qué hace el flow.
    Aparece cuando ejecutas: python flow.py --help
    """

    @step
    def start(self):
        """
        Paso obligatorio de inicio.
        Aquí normalmente se cargan datos o se configura el pipeline.
        """
        print("👋 ¡Hola desde el paso start!")
        
        # Asignamos datos a self → quedan disponibles en TODOS los pasos siguientes
        # Estos datos se llaman ARTIFACTS y Metaflow los persiste automáticamente
        self.mensaje = "Soy un artifact, vivo entre pasos"
        
        self.next(self.procesar)  # Le decimos a Metaflow cuál es el siguiente paso

    @step
    def procesar(self):
        """
        Paso intermedio.
        Podemos acceder a self.mensaje porque fue definido en start.
        """
        print(f"🔄 En el paso 'procesar'. Mensaje recibido: {self.mensaje}")
        self.resultado = self.mensaje.upper()  # Creamos un nuevo artifact
        self.next(self.end)

    @step
    def end(self):
        """
        Paso obligatorio de fin.
        Aquí normalmente se registran métricas finales o se hace cleanup.
        """
        print(f"✅ Flow terminado. Resultado: {self.resultado}")
        # end NO llama a self.next() — es el último paso

if __name__ == "__main__":
    HolaMetaflow()

In [ ]:
import os
os.makedirs("flows", exist_ok=True)

# Ejecutamos el flow desde el notebook
!python flows/01_hola_metaflow.py run

---
## 3. Concepto: Parameters (Parámetros de entrada)

Los **Parameters** permiten pasar argumentos al flow desde la línea de comandos  
(o desde el notebook). Son inmutables y están disponibles en todos los pasos.

In [ ]:
%%writefile flows/02_parameters.py
"""
CONCEPTO 2: Parameter

Parameter es como argparse pero integrado en Metaflow.
Se define a nivel de clase (no dentro de un método).
"""

from metaflow import FlowSpec, step, Parameter

class FlowConParametros(FlowSpec):

    # Definimos parámetros con tipo, descripción y valor por defecto
    test_size = Parameter(
        "test_size",
        help="Proporción del dataset para test (ej: 0.2 = 20%)",
        default=0.2,
        type=float
    )

    modelo = Parameter(
        "modelo",
        help="Tipo de modelo: 'rf' (Random Forest) o 'lr' (Logistic Regression)",
        default="rf"
    )

    semilla = Parameter(
        "semilla",
        help="Random seed para reproducibilidad",
        default=42,
        type=int
    )

    @step
    def start(self):
        # Los parámetros se acceden con self.nombre_parametro
        print(f"⚙️  Configuración recibida:")
        print(f"   - Test size:  {self.test_size}")
        print(f"   - Modelo:     {self.modelo}")
        print(f"   - Semilla:    {self.semilla}")
        self.next(self.end)

    @step
    def end(self):
        print("✅ Parámetros correctamente accesibles en todos los pasos")

if __name__ == "__main__":
    FlowConParametros()

In [ ]:
# Ejecutamos con parámetros por defecto
!python flows/02_parameters.py run

print("\n--- Ahora con parámetros personalizados ---\n")

# Sobreescribimos los valores por defecto
!python flows/02_parameters.py run --test_size 0.3 --modelo lr --semilla 123

---
## 4. Concepto: Branching (Ramas Paralelas)

Una de las funcionalidades más potentes de Metaflow.  
Puedes hacer que un paso se divida en **múltiples ramas paralelas** y luego **unirlas** (join).

```
            ┌──► rama_A ──┐
start ──►  split         join ──► end
            └──► rama_B ──┘
```

Perfecto para: entrenar múltiples modelos a la vez, probar hiperparámetros, etc.

In [ ]:
%%writefile flows/03_branching.py
"""
CONCEPTO 3: Branching (ramas paralelas) y Join

Usamos self.next() con múltiples argumentos para crear ramas.
En el paso join, usamos inputs para acceder a los resultados de cada rama.
"""

from metaflow import FlowSpec, step

class FlowConRamas(FlowSpec):

    @step
    def start(self):
        print("🚀 Inicio — vamos a entrenar 3 modelos en paralelo")
        # self.next con múltiples argumentos crea las ramas
        self.next(self.modelo_lr, self.modelo_rf, self.modelo_dt)

    @step
    def modelo_lr(self):
        """Rama 1: Simulamos entrenamiento de Logistic Regression"""
        import time
        print("   📊 Entrenando Logistic Regression...")
        time.sleep(0.1)  # Simulamos tiempo de entrenamiento
        self.nombre_modelo = "Logistic Regression"
        self.accuracy = 0.79
        self.next(self.join_modelos)

    @step
    def modelo_rf(self):
        """Rama 2: Simulamos entrenamiento de Random Forest"""
        import time
        print("   🌲 Entrenando Random Forest...")
        time.sleep(0.1)
        self.nombre_modelo = "Random Forest"
        self.accuracy = 0.83
        self.next(self.join_modelos)

    @step
    def modelo_dt(self):
        """Rama 3: Simulamos entrenamiento de Decision Tree"""
        import time
        print("   🌿 Entrenando Decision Tree...")
        time.sleep(0.1)
        self.nombre_modelo = "Decision Tree"
        self.accuracy = 0.76
        self.next(self.join_modelos)

    @step
    def join_modelos(self, inputs):
        """
        PASO JOIN: recibe 'inputs' (iterable con los resultados de cada rama)
        
        Regla importante: después del join, los artifacts de las ramas
        NO están disponibles directamente en self.
        Hay que extraerlos de 'inputs' manualmente.
        """
        print("\n🏆 Comparando modelos:")
        
        resultados = []
        for inp in inputs:
            resultados.append({
                "modelo": inp.nombre_modelo,
                "accuracy": inp.accuracy
            })
            print(f"   {inp.nombre_modelo}: accuracy = {inp.accuracy:.2%}")
        
        # Elegimos el mejor modelo
        mejor = max(resultados, key=lambda x: x["accuracy"])
        self.mejor_modelo = mejor["modelo"]
        self.mejor_accuracy = mejor["accuracy"]
        self.todos_resultados = resultados
        
        print(f"\n🥇 Mejor modelo: {self.mejor_modelo} ({self.mejor_accuracy:.2%})")
        
        self.next(self.end)

    @step
    def end(self):
        print(f"\n✅ Pipeline completado. Ganador: {self.mejor_modelo}")

if __name__ == "__main__":
    FlowConRamas()

In [ ]:
!python flows/03_branching.py run

---
## 5. Concepto: foreach (Branching Dinámico)

El `foreach` es branching dinámico: crea **una rama por cada elemento de una lista**.  
Ideal para grid search de hiperparámetros.

```
            ┌──► [params_1] ──┐
start ──►  split              join ──► end
            ├──► [params_2] ──┤
            └──► [params_3] ──┘
```

In [ ]:
%%writefile flows/04_foreach.py
"""
CONCEPTO 4: foreach — branching dinámico

self.next(self.paso, foreach='lista') crea una rama por cada
elemento de la lista. Dentro de cada rama, el elemento actual
está disponible en self.input
"""

from metaflow import FlowSpec, step

class GridSearchFlow(FlowSpec):

    @step
    def start(self):
        # Definimos los hiperparámetros a probar
        # Cada elemento será procesado en su propia rama
        self.parametros = [
            {"n_estimators": 50,  "max_depth": 3},
            {"n_estimators": 100, "max_depth": 5},
            {"n_estimators": 200, "max_depth": None},
            {"n_estimators": 100, "max_depth": 10},
        ]
        print(f"🔍 Iniciando grid search con {len(self.parametros)} configuraciones")
        
        # foreach crea una rama por cada elemento en self.parametros
        self.next(self.entrenar, foreach="parametros")

    @step
    def entrenar(self):
        """
        Este paso se ejecuta UNA VEZ POR CADA elemento de self.parametros.
        El elemento actual está en self.input (nombrado siempre así con foreach).
        """
        params = self.input  # El elemento actual del foreach
        print(f"   ⚙️  Probando: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}")
        
        # Simulamos un score (en el flow real del Titanic haremos el entrenamiento de verdad)
        import random
        random.seed(params["n_estimators"])
        self.score = round(random.uniform(0.78, 0.87), 4)
        self.params_usados = params
        
        self.next(self.join_resultados)

    @step
    def join_resultados(self, inputs):
        """Join de todas las ramas del foreach"""
        
        self.resultados = [
            {"params": inp.params_usados, "score": inp.score}
            for inp in inputs
        ]
        
        self.resultados.sort(key=lambda x: x["score"], reverse=True)
        
        print("\n📊 Resultados del Grid Search (ordenados por score):")
        for r in self.resultados:
            print(f"   Score: {r['score']:.4f} | Params: {r['params']}")
        
        self.mejores_params = self.resultados[0]["params"]
        self.next(self.end)

    @step
    def end(self):
        print(f"\n🏆 Mejores hiperparámetros encontrados: {self.mejores_params}")

if __name__ == "__main__":
    GridSearchFlow()

In [ ]:
!python flows/04_foreach.py run

---
## 6. Decoradores de Resiliencia: `@retry`, `@timeout`, `@catch`

Metaflow incluye decoradores para hacer los pasos más robustos en producción.

| Decorador | Uso |
|---|---|
| `@retry(times=3)` | Reintenta el paso N veces si falla |
| `@timeout(seconds=60)` | Mata el paso si tarda más de N segundos |
| `@catch(var='error')` | Captura excepciones sin detener el flow |

In [ ]:
%%writefile flows/05_decoradores_resiliencia.py
"""
CONCEPTO 5: @retry, @timeout, @catch

Estos decoradores hacen los pasos tolerantes a fallos.
Son especialmente útiles en entornos cloud con recursos efímeros.
"""

from metaflow import FlowSpec, step, retry, timeout, catch

class FlowRobusto(FlowSpec):

    @step
    def start(self):
        print("🚀 Iniciando flow robusto")
        self.intento = 0
        self.next(self.paso_con_retry)

    @retry(times=3, minutes_between_retries=0)
    @step
    def paso_con_retry(self):
        """
        @retry(times=3) → si este paso lanza una excepción,
        Metaflow lo reintentará hasta 3 veces antes de fallar definitivamente.
        
        Útil para: llamadas a APIs externas, descargas de datos, etc.
        """
        import random
        print("   🔄 Ejecutando paso con retry...")
        # Simulamos un fallo aleatorio (50% de probabilidad)
        # En producción esto podría ser un timeout de red, etc.
        if random.random() < 0.3:  # 30% de fallo
            raise RuntimeError("💥 Fallo simulado — Metaflow reintentará")
        print("   ✅ Paso ejecutado con éxito")
        self.next(self.paso_con_timeout)

    @timeout(seconds=10)
    @step
    def paso_con_timeout(self):
        """
        @timeout(seconds=10) → si el paso tarda más de 10 segundos,
        Metaflow lo interrumpe y lanza un TimeoutException.
        
        Útil para evitar que un paso colgado bloquee todo el pipeline.
        """
        import time
        print("   ⏱️  Paso con timeout (máx 10s)...")
        time.sleep(1)  # Tardamos 1 segundo (dentro del límite)
        print("   ✅ Completado dentro del tiempo límite")
        self.next(self.paso_con_catch)

    @catch(var="mi_error", print_exception=True)
    @step
    def paso_con_catch(self):
        """
        @catch(var='mi_error') → si hay una excepción, en vez de parar el flow,
        la guarda en self.mi_error y continúa.
        
        self.mi_error será None si no hubo error,
        o la excepción capturada si la hubo.
        """
        print("   🕵️  Paso con catch...")
        raise ValueError("Este error será capturado, no detendrá el flow")

    @step
    def end(self):
        # Comprobamos si @catch capturó algo
        if self.mi_error is not None:
            print(f"⚠️  El paso anterior falló pero el flow siguió: {self.mi_error}")
        else:
            print("✅ Todo fue bien")
        print("🏁 Flow completado")

if __name__ == "__main__":
    FlowRobusto()

In [ ]:
!python flows/05_decoradores_resiliencia.py run

---
## 7. Concepto: `current` — Metadata del Run en Curso

El objeto `current` te da acceso a metadata del run que está ejecutándose ahora mismo.  
Es de solo lectura y está disponible dentro de cualquier `@step`.

In [ ]:
%%writefile flows/06_current.py
"""
CONCEPTO 6: current

El objeto 'current' contiene información sobre el run actual.
Muy útil para logging, nombrar modelos, etc.
"""

from metaflow import FlowSpec, step, current

class FlowConCurrent(FlowSpec):

    @step
    def start(self):
        # current está disponible dentro de los pasos
        print("📋 Información del run actual:")
        print(f"   Flow name:   {current.flow_name}")
        print(f"   Run ID:      {current.run_id}")
        print(f"   Step name:   {current.step_name}")
        print(f"   Task ID:     {current.task_id}")
        print(f"   Pathspec:    {current.pathspec}")
        # pathspec = FlowName/RunID/StepName/TaskID — identificador único
        
        # Guardamos el run_id para usarlo como identificador del modelo
        self.model_id = f"titanic_model_run_{current.run_id}"
        self.next(self.end)

    @step
    def end(self):
        print(f"\n🆔 ID del modelo generado: {self.model_id}")
        print(f"   (En producción, usaríamos esto para nombrar el modelo en el registry)")

if __name__ == "__main__":
    FlowConCurrent()

In [ ]:
!python flows/06_current.py run

---
## 8. La Client API: Inspeccionando Runs desde el Notebook

Una vez que has ejecutado flows, puedes **introspectarlos desde Python** usando la Client API.  
Esto permite comparar experimentos, recuperar modelos, analizar resultados históricos, etc.

In [ ]:
"""
CONCEPTO 7: Client API

La Client API permite acceder a los runs pasados desde código Python.
Útil para análisis post-ejecución, comparación de experimentos, etc.
"""

from metaflow import Flow, Run, namespace

# namespace(None) → accede a todos los runs (sin filtrar por usuario)
namespace(None)

# ── Listar todos los flows disponibles
from metaflow import Metaflow
mf = Metaflow()
print("📦 Flows disponibles en el metadata store:")
for flow in mf.flows:
    print(f"   - {flow.id}")

In [ ]:
# Accedemos al flow de branching que ejecutamos antes
try:
    flow = Flow("FlowConRamas")
    
    print(f"\n🔍 Inspeccionando: {flow.id}")
    print(f"   Último run: {flow.latest_run}")
    
    ultimo_run = flow.latest_run
    print(f"   ID del run: {ultimo_run.id}")
    print(f"   Estado:     {ultimo_run.successful}")
    
    # Accedemos a los pasos del run
    print(f"\n   Pasos ejecutados:")
    for step in ultimo_run.steps():
        print(f"   └─ {step.id}")
    
    # Recuperamos artifacts del paso end
    end_step = ultimo_run["end"]
    task = list(end_step.tasks())[0]
    
    print(f"\n   🏆 Mejor modelo (artifact recuperado): {task.data.mejor_modelo}")
    print(f"   📊 Accuracy: {task.data.mejor_accuracy:.2%}")
    
except Exception as e:
    print(f"ℹ️  {e}")
    print("Asegúrate de haber ejecutado la celda del Flow 03 primero.")

---
## 9. ⭐ EL FLOW PRINCIPAL: TitanicMLFlow

Ahora juntamos **todos los conceptos** en un pipeline real de ML sobre el dataset del Titanic.

**Pipeline completo:**
```
start
  │
  ▼
cargar_datos
  │
  ▼
preprocesar
  │
  ├──► entrenar_rf ──┐
  ├──► entrenar_lr ──┤
  └──► entrenar_gb ──┘
             │
             ▼
         comparar_modelos
             │
             ▼
         evaluar_final
             │
             ▼
            end
```

In [ ]:
%%writefile flows/07_titanic_ml_flow.py
"""
FLOW PRINCIPAL: TitanicMLFlow

Pipeline completo de ML que integra todos los conceptos de Metaflow:
- Parameters
- Branching estático (3 modelos en paralelo)
- Join
- @retry para carga de datos
- @catch para entrenamiento
- current para metadata
- Artifacts (modelo serializado, métricas, etc.)
"""

from metaflow import FlowSpec, step, Parameter, retry, catch, current
import json

class TitanicMLFlow(FlowSpec):
    """
    Pipeline completo de ML para predecir supervivencia en el Titanic.
    Entrena Random Forest, Logistic Regression y Gradient Boosting
    en paralelo y selecciona el mejor modelo.
    """

    # ── Parámetros configurables ──────────────────────────────────────────
    test_size = Parameter(
        "test_size",
        help="Proporción del dataset para test",
        default=0.2,
        type=float
    )

    semilla = Parameter(
        "semilla",
        help="Semilla aleatoria para reproducibilidad",
        default=42,
        type=int
    )

    n_estimators = Parameter(
        "n_estimators",
        help="Número de árboles para Random Forest y Gradient Boosting",
        default=100,
        type=int
    )

    # ── Paso 1: Inicio ────────────────────────────────────────────────────
    @step
    def start(self):
        """
        Punto de entrada del pipeline.
        Registramos la configuración usando 'current'.
        """
        print("🚢 ===== TitanicMLFlow =====")
        print(f"   Run ID:      {current.run_id}")
        print(f"   Pathspec:    {current.pathspec}")
        print(f"   Test size:   {self.test_size}")
        print(f"   Semilla:     {self.semilla}")
        print(f"   n_estimators:{self.n_estimators}")
        
        self.run_id = current.run_id
        self.next(self.cargar_datos)

    # ── Paso 2: Carga de datos (con retry por si falla la descarga) ────────
    @retry(times=3, minutes_between_retries=0)
    @step
    def cargar_datos(self):
        """
        Descargamos el dataset del Titanic.
        Usamos @retry porque la descarga podría fallar por red.
        """
        import pandas as pd
        
        print("📥 Cargando dataset del Titanic...")
        
        # Intentamos cargar desde seaborn (fuente online)
        try:
            import seaborn as sns
            df = sns.load_dataset("titanic")
        except Exception:
            # Fallback: URL directa
            url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
            df = pd.read_csv(url)
        
        print(f"   ✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
        print(f"   Columnas: {list(df.columns)}")
        print(f"   Supervivientes: {df['survived'].sum()} ({df['survived'].mean():.1%})")
        
        # Guardamos el dataframe como artifact
        # Metaflow lo serializa automáticamente (usa pickle internamente)
        self.df_raw = df
        self.n_samples = len(df)
        
        self.next(self.preprocesar)

    # ── Paso 3: Preprocesamiento ───────────────────────────────────────────
    @step
    def preprocesar(self):
        """
        Feature engineering y división train/test.
        """
        import pandas as pd
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import LabelEncoder
        
        print("🔧 Preprocesando datos...")
        
        df = self.df_raw.copy()
        
        # ── Feature Engineering ──────────────────────────────────────────
        
        # 1. Seleccionamos features relevantes
        features_usar = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
        target = "survived"
        
        # 2. Rellenamos valores nulos
        df["age"] = df["age"].fillna(df["age"].median())
        df["fare"] = df["fare"].fillna(df["fare"].median())
        df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
        
        # 3. Codificamos variables categóricas
        df["sex"] = LabelEncoder().fit_transform(df["sex"])  # male=1, female=0
        df["embarked"] = LabelEncoder().fit_transform(df["embarked"].astype(str))
        
        # 4. Feature adicional: tamaño de la familia
        df["family_size"] = df["sibsp"] + df["parch"]
        features_usar.append("family_size")
        
        X = df[features_usar]
        y = df[target]
        
        # 5. División train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.semilla,
            stratify=y  # Mantenemos proporción de clases
        )
        
        print(f"   Train: {len(X_train)} muestras | Test: {len(X_test)} muestras")
        print(f"   Features: {list(X.columns)}")
        
        # Guardamos como artifacts
        self.X_train = X_train
        self.X_test  = X_test
        self.y_train = y_train
        self.y_test  = y_test
        self.feature_names = list(X.columns)
        
        # Ahora lanzamos 3 ramas en paralelo
        self.next(self.entrenar_rf, self.entrenar_lr, self.entrenar_gb)

    # ── Paso 4a: Random Forest ─────────────────────────────────────────────
    @step
    def entrenar_rf(self):
        """
        Rama 1: Random Forest Classifier
        """
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
        
        print("🌲 Entrenando Random Forest...")
        
        modelo = RandomForestClassifier(
            n_estimators=self.n_estimators,
            random_state=self.semilla,
            max_depth=10
        )
        modelo.fit(self.X_train, self.y_train)
        
        y_pred = modelo.predict(self.X_test)
        y_proba = modelo.predict_proba(self.X_test)[:, 1]
        
        self.nombre_modelo = "Random Forest"
        self.modelo = modelo  # El modelo serializado como artifact
        self.accuracy = accuracy_score(self.y_test, y_pred)
        self.auc     = roc_auc_score(self.y_test, y_proba)
        self.f1      = f1_score(self.y_test, y_pred)
        
        # Feature importance
        self.feature_importance = dict(zip(
            self.feature_names,
            modelo.feature_importances_.tolist()
        ))
        
        print(f"   Accuracy: {self.accuracy:.4f} | AUC: {self.auc:.4f} | F1: {self.f1:.4f}")
        self.next(self.comparar_modelos)

    # ── Paso 4b: Logistic Regression ──────────────────────────────────────
    @step
    def entrenar_lr(self):
        """
        Rama 2: Logistic Regression (con escalado de features)
        """
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
        
        print("📈 Entrenando Logistic Regression...")
        
        # Usamos un Pipeline de sklearn para encapsular scaler + modelo
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=1000, random_state=self.semilla))
        ])
        pipeline.fit(self.X_train, self.y_train)
        
        y_pred = pipeline.predict(self.X_test)
        y_proba = pipeline.predict_proba(self.X_test)[:, 1]
        
        self.nombre_modelo = "Logistic Regression"
        self.modelo = pipeline
        self.accuracy = accuracy_score(self.y_test, y_pred)
        self.auc     = roc_auc_score(self.y_test, y_proba)
        self.f1      = f1_score(self.y_test, y_pred)
        self.feature_importance = None
        
        print(f"   Accuracy: {self.accuracy:.4f} | AUC: {self.auc:.4f} | F1: {self.f1:.4f}")
        self.next(self.comparar_modelos)

    # ── Paso 4c: Gradient Boosting ─────────────────────────────────────────
    @step
    def entrenar_gb(self):
        """
        Rama 3: Gradient Boosting Classifier
        """
        from sklearn.ensemble import GradientBoostingClassifier
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
        
        print("🚀 Entrenando Gradient Boosting...")
        
        modelo = GradientBoostingClassifier(
            n_estimators=self.n_estimators,
            random_state=self.semilla,
            max_depth=4
        )
        modelo.fit(self.X_train, self.y_train)
        
        y_pred = modelo.predict(self.X_test)
        y_proba = modelo.predict_proba(self.X_test)[:, 1]
        
        self.nombre_modelo = "Gradient Boosting"
        self.modelo = modelo
        self.accuracy = accuracy_score(self.y_test, y_pred)
        self.auc     = roc_auc_score(self.y_test, y_proba)
        self.f1      = f1_score(self.y_test, y_pred)
        self.feature_importance = dict(zip(
            self.feature_names,
            modelo.feature_importances_.tolist()
        ))
        
        print(f"   Accuracy: {self.accuracy:.4f} | AUC: {self.auc:.4f} | F1: {self.f1:.4f}")
        self.next(self.comparar_modelos)

    # ── Paso 5: Join — comparar modelos ────────────────────────────────────
    @step
    def comparar_modelos(self, inputs):
        """
        Join de las 3 ramas de entrenamiento.
        Comparamos métricas y seleccionamos el mejor modelo.
        """
        print("\n📊 === Comparativa de Modelos ===")
        print(f"{'Modelo':<25} {'Accuracy':>10} {'AUC':>8} {'F1':>8}")
        print("-" * 55)
        
        resultados = []
        for inp in inputs:
            resultados.append({
                "nombre": inp.nombre_modelo,
                "modelo": inp.modelo,
                "accuracy": inp.accuracy,
                "auc": inp.auc,
                "f1": inp.f1,
                "feature_importance": inp.feature_importance,
            })
            print(f"{inp.nombre_modelo:<25} {inp.accuracy:>10.4f} {inp.auc:>8.4f} {inp.f1:>8.4f}")
        
        # Seleccionamos el mejor modelo por AUC (métrica más robusta)
        mejor = max(resultados, key=lambda x: x["auc"])
        
        print(f"\n🏆 Mejor modelo (por AUC): {mejor['nombre']}")
        
        # Guardamos todo como artifacts
        self.todos_resultados = resultados  
        self.mejor_nombre     = mejor["nombre"]
        self.mejor_modelo     = mejor["modelo"]  # El modelo sklearn serializado
        self.mejor_accuracy   = mejor["accuracy"]
        self.mejor_auc        = mejor["auc"]
        self.mejor_f1         = mejor["f1"]
        self.mejor_fi         = mejor["feature_importance"]
        
        self.next(self.evaluar_final)

    # ── Paso 6: Evaluación final ───────────────────────────────────────────
    @step
    def evaluar_final(self):
        """
        Evaluación detallada del mejor modelo.
        Generamos reporte de clasificación y matriz de confusión.
        """
        from sklearn.metrics import classification_report, confusion_matrix
        import numpy as np
        
        print(f"\n🔬 Evaluación detallada de: {self.mejor_nombre}")
        
        y_pred = self.mejor_modelo.predict(self.X_test)
        
        # Reporte de clasificación
        report = classification_report(
            self.y_test, y_pred,
            target_names=["No sobrevivió", "Sobrevivió"]
        )
        print("\nReporte de Clasificación:")
        print(report)
        
        # Matriz de confusión
        cm = confusion_matrix(self.y_test, y_pred)
        print("Matriz de Confusión:")
        print(f"   TN={cm[0,0]}  FP={cm[0,1]}")
        print(f"   FN={cm[1,0]}  TP={cm[1,1]}")
        
        # Feature importance (si aplica)
        if self.mejor_fi:
            fi_sorted = sorted(self.mejor_fi.items(), key=lambda x: x[1], reverse=True)
            print("\nImportancia de Features (top):")
            for feat, imp in fi_sorted:
                bar = "█" * int(imp * 40)
                print(f"   {feat:<15} {imp:.4f} {bar}")
        
        # Guardamos el reporte como artifact de texto
        self.classification_report = report
        self.confusion_matrix = cm.tolist()
        
        self.next(self.end)

    # ── Paso 7: End ───────────────────────────────────────────────────────
    @step
    def end(self):
        """
        Resumen final del pipeline.
        """
        print("\n" + "="*50)
        print("✅ TitanicMLFlow completado exitosamente")
        print("="*50)
        print(f"   Run ID:        {self.run_id}")
        print(f"   Mejor modelo:  {self.mejor_nombre}")
        print(f"   Accuracy:      {self.mejor_accuracy:.4f}")
        print(f"   AUC:           {self.mejor_auc:.4f}")
        print(f"   F1 Score:      {self.mejor_f1:.4f}")
        print("\n💾 Todos los artifacts están persistidos en Metaflow.")
        print("   Puedes recuperarlos con la Client API desde cualquier notebook.")

if __name__ == "__main__":
    TitanicMLFlow()

In [ ]:
# ¡Ejecutamos el flow principal!
# Este comando corre el pipeline completo con los parámetros por defecto
!python flows/07_titanic_ml_flow.py run

In [ ]:
# Ejecutamos de nuevo con parámetros diferentes para comparar
!python flows/07_titanic_ml_flow.py run --test_size 0.25 --n_estimators 200 --semilla 99

---
## 10. Client API Avanzada: Comparando Experimentos

Ahora que tenemos varios runs del TitanicMLFlow, usamos la Client API para compararlos.

In [ ]:
"""
CONCEPTO 8: Client API avanzada — comparación de experimentos

Iteramos sobre todos los runs de un flow y comparamos sus métricas.
Esto es como un MLflow tracking pero nativo en Metaflow.
"""

from metaflow import Flow, namespace
import pandas as pd

namespace(None)

try:
    flow = Flow("TitanicMLFlow")
    
    print(f"📊 Comparando todos los runs de TitanicMLFlow:")
    print(f"   Total de runs: {len(list(flow.runs()))}")
    print()
    
    comparativa = []
    
    for run in flow.runs():
        if not run.successful:
            continue
        
        # Accedemos al paso 'end' para obtener los artifacts finales
        try:
            end_task = list(run["end"].tasks())[0]
            data = end_task.data
            
            comparativa.append({
                "Run ID":       run.id,
                "Mejor Modelo": data.mejor_nombre,
                "Accuracy":     round(data.mejor_accuracy, 4),
                "AUC":          round(data.mejor_auc, 4),
                "F1":           round(data.mejor_f1, 4),
                "Test Size":    data.test_size,
                "Semilla":      data.semilla,
            })
        except Exception:
            pass
    
    if comparativa:
        df_comp = pd.DataFrame(comparativa)
        df_comp = df_comp.sort_values("AUC", ascending=False)
        print(df_comp.to_string(index=False))
        
        mejor_run = df_comp.iloc[0]
        print(f"\n🏆 Mejor run: {mejor_run['Run ID']} (AUC = {mejor_run['AUC']})")
    else:
        print("No hay runs completados aún.")

except Exception as e:
    print(f"Ejecuta primero el TitanicMLFlow. Error: {e}")

---
## 11. Recuperar y Usar el Mejor Modelo (Inference)

Una vez entrenado, podemos recuperar el modelo de Metaflow y usarlo para hacer predicciones sin necesidad de reentrenar.

In [ ]:
"""
CONCEPTO 9: Recuperar artifacts para inferencia

El modelo entrenado está persistido como artifact.
Lo recuperamos con la Client API y lo usamos directamente.
"""

from metaflow import Flow, namespace
import pandas as pd
import numpy as np

namespace(None)

try:
    flow = Flow("TitanicMLFlow")
    ultimo_run = flow.latest_successful_run
    
    # Recuperamos el modelo del último run exitoso
    end_task = list(ultimo_run["end"].tasks())[0]
    modelo = end_task.data.mejor_modelo
    feature_names = end_task.data.feature_names
    mejor_nombre = end_task.data.mejor_nombre
    
    print(f"✅ Modelo cargado desde Run {ultimo_run.id}: {mejor_nombre}")
    print(f"   Features esperadas: {feature_names}")
    
    # ── Simulamos nuevos pasajeros para predecir ──────────────────────────
    nuevos_pasajeros = pd.DataFrame([
        # pclass  sex  age  sibsp  parch  fare  embarked  family_size
        {"pclass": 1, "sex": 0, "age": 28, "sibsp": 1, "parch": 0, "fare": 100, "embarked": 2, "family_size": 1},
        {"pclass": 3, "sex": 1, "age": 22, "sibsp": 0, "parch": 0, "fare": 7.25, "embarked": 2, "family_size": 0},
        {"pclass": 2, "sex": 0, "age": 35, "sibsp": 0, "parch": 2, "fare": 26, "embarked": 2, "family_size": 2},
        {"pclass": 1, "sex": 1, "age": 45, "sibsp": 1, "parch": 1, "fare": 150, "embarked": 0, "family_size": 2},
    ])
    
    predicciones = modelo.predict(nuevos_pasajeros[feature_names])
    probabilidades = modelo.predict_proba(nuevos_pasajeros[feature_names])[:, 1]
    
    print("\n🔮 Predicciones para nuevos pasajeros:")
    print(f"{'Pasajero':<12} {'Clase':<8} {'Sexo':<8} {'Edad':<8} {'Predicción':<15} {'Prob. Supervivencia'}")
    print("-" * 70)
    
    sexo_map = {0: "Mujer", 1: "Hombre"}
    pred_map = {0: "❌ No sobrevive", 1: "✅ Sobrevive"}
    
    for i, (pred, prob) in enumerate(zip(predicciones, probabilidades)):
        row = nuevos_pasajeros.iloc[i]
        print(f"{i+1:<12} {int(row.pclass):<8} {sexo_map[int(row.sex)]:<8} {int(row.age):<8} {pred_map[pred]:<15} {prob:.1%}")

except Exception as e:
    print(f"Ejecuta primero el TitanicMLFlow. Error: {e}")

---
## 12. Concepto: `@environment` y `@pypi` (Gestión de Dependencias)

Metaflow permite especificar dependencias **por paso**. En cloud, cada paso puede tener su propio entorno Python.

In [ ]:
%%writefile flows/08_dependencias.py
"""
CONCEPTO 10: @pypi y @environment

@pypi(packages={"nombre": "version"}) instala paquetes específicos para ese paso.
@environment(vars={"VARIABLE": "valor"}) inyecta variables de entorno.

Esto es especialmente útil cuando distintos pasos necesitan librerías
diferentes o versiones específicas (p.ej., un paso usa torch, otro usa jax).
"""

from metaflow import FlowSpec, step, pypi, environment

class FlowConDependencias(FlowSpec):

    @step
    def start(self):
        print("🚀 Inicio")
        self.next(self.paso_con_pypi)

    # Este paso instala una versión específica de pandas solo para él
    # En local: usa el entorno ya instalado. En cloud (AWS Batch/K8s): crea env aislado
    @pypi(packages={"pandas": "2.0.0"})
    @step
    def paso_con_pypi(self):
        import pandas as pd
        print(f"   📦 Versión de pandas en este paso: {pd.__version__}")
        self.next(self.paso_con_env)

    # Este paso tiene acceso a variables de entorno específicas
    @environment(vars={"MI_ENTORNO": "produccion", "MODELO_VERSION": "v1.2"})
    @step
    def paso_con_env(self):
        import os
        print(f"   🌐 Entorno: {os.environ.get('MI_ENTORNO')}")
        print(f"   📌 Versión del modelo: {os.environ.get('MODELO_VERSION')}")
        self.next(self.end)

    @step
    def end(self):
        print("✅ Completado")

if __name__ == "__main__":
    FlowConDependencias()

---
## 13. Concepto: `@resources` (Escalado en Cloud)

Cuando ejecutas flows en AWS Batch o Kubernetes, `@resources` te permite asignar CPU y memoria a cada paso.

```python
from metaflow import resources

@resources(cpu=4, memory=16000)  # 4 CPUs, 16GB RAM
@step
def entrenar_modelo_grande(self):
    # Este paso se ejecutará en una máquina con 4 CPUs y 16GB
    ...

@resources(gpu=1, memory=32000)  # 1 GPU, 32GB RAM
@step  
def fine_tune_llm(self):
    # Para pasos que necesitan GPU
    ...
```

> 💡 En ejecución local, `@resources` se ignora. Solo tiene efecto en ejecución cloud (`--with batch` o `--with kubernetes`).

**Para deployar en AWS:**
```bash
python flow.py run --with batch
```

---
## 14. Concepto: Cards (Visualizaciones Automáticas)

Las **Cards** son HTML interactivos generados automáticamente por Metaflow que documentan lo que hizo cada paso. Son perfectas para crear reportes de experimentos.

In [ ]:
%%writefile flows/09_cards.py
"""
CONCEPTO 11: @card — visualizaciones automáticas de runs

@card genera un HTML con información del paso.
Puedes añadir tablas, imágenes, texto y gráficas.
"""

from metaflow import FlowSpec, step, card
from metaflow.cards import Table, Markdown, Image

class TitanicReportFlow(FlowSpec):

    @card  # Genera una card básica con info del step
    @step
    def start(self):
        import seaborn as sns
        self.df = sns.load_dataset("titanic")
        self.next(self.analizar)

    @card(type="blank")  # Card en blanco donde añadimos contenido personalizado
    @step
    def analizar(self):
        from metaflow.cards import current as card_current
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import io, base64
        
        # Añadimos contenido a la card
        card_current.add(Markdown("# 🚢 Análisis del Titanic"))
        card_current.add(Markdown(f"**Total de pasajeros:** {len(self.df)}"))
        card_current.add(Markdown(f"**Tasa de supervivencia:** {self.df['survived'].mean():.1%}"))
        
        # Tabla de resumen por clase
        resumen = self.df.groupby("pclass")["survived"].agg(["count", "mean"]).reset_index()
        resumen.columns = ["Clase", "Total", "Tasa Supervivencia"]
        resumen["Tasa Supervivencia"] = resumen["Tasa Supervivencia"].apply(lambda x: f"{x:.1%}")
        
        card_current.add(Markdown("## Supervivencia por Clase"))
        card_current.add(Table(resumen.values.tolist(), headers=list(resumen.columns)))
        
        self.next(self.end)

    @step
    def end(self):
        print("✅ Flow completado. Visualiza la card con:")
        print("   python flows/09_cards.py card show --origin analizar")

if __name__ == "__main__":
    TitanicReportFlow()

---
## 15. Referencia Rápida: Comandos CLI de Metaflow

```bash
# Ejecutar un flow
python flow.py run

# Ejecutar con parámetros
python flow.py run --param_name valor

# Ver ayuda de un flow
python flow.py --help
python flow.py run --help

# Ver logs de la última ejecución
python flow.py logs --steps paso_nombre

# Inspeccionar artifacts de un run específico
python flow.py dump --run-id 3

# Ver cards generadas
python flow.py card view --origin nombre_paso

# Ejecutar solo desde un paso específico (resume)
python flow.py resume --origin nombre_paso

# Ejecutar en AWS Batch
python flow.py run --with batch

# Ejecutar en Kubernetes
python flow.py run --with kubernetes
```

---
## 16. Resumen Conceptual Final

### Los 3 conceptos más importantes de Metaflow

| Concepto | En código | ¿Para qué? |
|---|---|---|
| **Flow** | `class MiFlow(FlowSpec)` | Define el pipeline completo |
| **Step** | `@step` en cada método | Define una unidad de trabajo atómica |
| **Artifact** | `self.variable = valor` | Persiste datos entre pasos y entre runs |

### ¿Por qué usar Metaflow en vez de scripts sueltos?

| Sin Metaflow | Con Metaflow |
|---|---|
| "¿Con qué datos entrené el modelo de ayer?" | Todo versionado automáticamente |
| Copiar/pegar código entre notebooks | Flows reutilizables con Parameters |
| "El script de preprocesamiento tardó 2h, tengo que volver a esperar" | `resume` desde cualquier paso |
| Difícil escalar a cloud | `--with batch` → AWS Batch en una línea |
| Sin paralelización | Branching nativo |

### Decoradores de un vistazo

```python
@step                          # Obligatorio en cada paso
@retry(times=3)                # Reintentar N veces si falla
@timeout(seconds=60)           # Límite de tiempo
@catch(var="err")              # Capturar excepciones
@resources(cpu=4, memory=8000) # Recursos en cloud
@pypi(packages={"lib": "1.0"}) # Dependencias por paso
@environment(vars={"K": "V"})  # Variables de entorno
@card                          # Generar reporte visual
```

---
> 📚 **Más recursos:**
> - Documentación oficial: https://docs.metaflow.org
> - Tutorials interactivos: https://outerbounds.com/docs/tutorials/
> - GitHub: https://github.com/Netflix/metaflow